In [3]:
import numpy as np
import pandas as pd
import pandas_datareader.data as web
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import yfinance as yf
import os
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler
from datetime import timedelta
import time
import requests

# Link to your custom model file
from src.models.stock_lstm import StockLSTM

%matplotlib inline
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# S&P 500 Ticker List - Comprehensive
sp500_tickers = [
    "MMM", "AOS", "ABT", "ABBV", "ACN", "ADBE", "AMD", "AES", "AFL", "A",
    "APD", "ABNB", "AKAM", "ALB", "ARE", "ALGN", "ALLE", "LNT", "ALL", "GOOGL",
    "GOOG", "MO", "AMZN", "AMCR", "AEE", "AEP", "AXP", "AIG", "AMT", "AWK",
    "AMP", "AME", "AMGN", "APH", "ADI", "AON", "APA", "APO", "AAPL", "AMAT",
    "APP", "APTV", "ACGL", "ADM", "ARES", "ANET", "AJG", "AIZ", "T", "ATO",
    "ADSK", "ADP", "AZO", "AVB", "AVY", "AXON", "BKR", "BALL", "BAC", "BAX",
    "BDX", "BRK.B", "BBY", "TECH", "BIIB", "BLK", "BX", "BK", "BA", "BKNG",
    "BSX", "BMY", "AVGO", "BR", "BRO", "BF.B", "BLDR", "BG", "BXP", "CHRW",
    "CDNS", "CPT", "CPB", "COF", "CAH", "CCL", "CARR", "CVNA", "CAT", "CBOE",
    "CBRE", "CDW", "COR", "CNC", "CNP", "CF", "CRL", "SCHW", "CHTR", "CVX",
    "CMG", "CB", "CHD", "CI", "CINF", "CTAS", "CSCO", "C", "CFG", "CLX",
    "CME", "CMS", "KO", "CTSH", "COIN", "CL", "CMCSA", "FIX", "CAG", "COP",
    "ED", "STZ", "CEG", "COO", "CPRT", "GLW", "CPAY", "CTVA", "CSGP", "COST",
    "CTRA", "CRH", "CRWD", "CCI", "CSX", "CMI", "CVS", "DHR", "DRI", "DDOG",
    "DVA", "DAY", "DECK", "DE", "DELL", "DAL", "DVN", "DXCM", "FANG", "DLR",
    "DG", "DLTR", "D", "DPZ", "DASH", "DOV", "DOW", "DHI", "DTE", "DUK",
    "DD", "ETN", "EBAY", "ECL", "EIX", "EW", "EA", "ELV", "EME", "EMR",
    "ETR", "EOG", "EPAM", "EQT", "EFX", "EQIX", "EQR", "ERIE", "ESS", "EL",
    "EG", "EVRG", "ES", "EXC", "EXPE", "EXPD", "EXR", "XOM", "FFIV", "FDS",
    "FICO", "FAST", "FRT", "FDX", "FIS", "FITB", "FSLR", "FE", "FISV", "F",
    "FTNT", "FTV", "FOXA", "FOX", "BEN", "FCX", "GRMN", "IT", "GE", "GEHC",
    "GEV", "GEN", "GNRC", "GD", "GIS", "GM", "GPC", "GILD", "GPN", "GL",
    "GDDY", "GS", "HAL", "HIG", "HAS", "HCA", "DOC", "HSIC", "HSY", "HES",
    "HPE", "HLT", "HOLX", "HD", "HON", "HRL", "HST", "HWM", "HPQ", "HUBB",
    "HUM", "HBAN", "HII", "IBM", "IEX", "IDXX", "ITW", "ILMN", "INCY", "IR",
    "PODD", "INTC", "ICE", "IFF", "IP", "IPG", "INTU", "ISRG", "IVZ", "INVH",
    "IQV", "IRM", "JBHT", "JBL", "JKHY", "J", "JNJ", "JCI", "JPM", "JNPR",
    "K", "KVUE", "KDP", "KEY", "KEYS", "KMB", "KIM", "KMI", "KLAC", "KNAP",
    "KHC", "KR", "LHX", "LH", "LRCX", "LW", "LVS", "LDOS", "LEN", "LIN",
    "LYV", "LKQ", "LMT", "L", "LOW", "LULU", "LYB", "MTB", "MRO", "MPC",
    "MKTX", "MAR", "MMC", "MLM", "MAS", "MA", "MTCH", "MKC", "MCD", "MCK",
    "MDT", "MRK", "META", "MET", "MTD", "MGM", "MCHP", "MU", "MSFT", "MAA",
    "MRNA", "MHK", "MOH", "TAP", "MDLZ", "MPWR", "MNST", "MCO", "MS", "MOS",
    "MSI", "MSCI", "NDAQ", "NTAP", "NFLX", "NEM", "NWL", "NEM", "NWSA", "NWS",
    "NEE", "NKE", "NI", "NDSN", "NSC", "NTRS", "NOC", "NCLH", "NRG", "NUE",
    "NVDA", "NVR", "NXPI", "ORLY", "OXY", "ODFL", "OMC", "ON", "OKE", "ORCL",
    "OTIS", "PCAR", "PKG", "PANW", "PARA", "PH", "PAYX", "PAYC", "PYPL", "PNR",
    "PEP", "PFE", "PCG", "PM", "PSX", "PNW", "PNC", "POOL", "PPG", "PPL",
    "PFG", "PG", "PGR", "PLD", "PRU", "PEG", "PTC", "PSA", "PHM", "QRVO",
    "PWR", "QCOM", "DGX", "RL", "RJF", "RTX", "O", "REG", "REGN", "RF",
    "RSG", "RMD", "RVTY", "ROK", "ROL", "ROP", "ROST", "RCL", "SPGI", "CRM",
    "SBAC", "SLB", "STX", "SRE", "NOW", "SHW", "SPG", "SWKS", "SJM", "SNA",
    "SOLV", "SO", "LUV", "SWK", "SBUX", "STT", "STLD", "STE", "SYK", "SYF",
    "SNPS", "SYY", "TMUS", "TROW", "TTWO", "TPR", "TRGP", "TGT", "TEL", "TDY",
    "TFX", "TER", "TSLA", "TXN", "TXT", "TMO", "TJX", "TSG", "TRV", "TRMB",
    "TFC", "TYL", "TSN", "USB", "UBER", "UDR", "ULTA", "UNP", "UAL", "UPS",
    "URI", "UNH", "UHS", "VLO", "VTR", "VLTO", "VRSN", "VRSK", "VZ", "VRTX",
    "V", "VICI", "VMC", "WRB", "GWW", "WAB", "WBA", "WMT", "DIS", "WBD",
    "WM", "WAT", "WEC", "WFC", "WELL", "WST", "WDC", "WRK", "WY", "WHR",
    "WMB", "WTW", "WYNN", "XEL", "XYL", "YUM", "ZBRA", "ZBH", "ZTS",
    "GEHC", "GEV", "GEN", "GNRC", "GD", "GIS", "GM", "GPC", "GILD"
]


print(len(sp500_tickers))

508


In [6]:

# 1. Setup Directory (Relative to 'Notebooks' folder)
data_dir = os.path.join('..', 'Data')
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

print(f"Starting batch download of {len(sp500_tickers)} stocks via Stooq...")
print(f"Data will be saved to: {os.path.abspath(data_dir)}")

# 2. The Download Loop
success_count = 0
failed_tickers = []

for ticker in sp500_tickers:
    # Standardize the file name for your local Data folder
    file_path = os.path.join(data_dir, f"{ticker}.csv")

    # Skip if we already have it to avoid redundant API calls
    if os.path.exists(file_path):
        continue

    try:
        # Format for Stooq: Dots to Dashes (e.g., BRK.B -> BRK-B) and add .US suffix
        stooq_ticker = f"{ticker.replace('.', '-')}.US"

        # Download data starting from 4 years ago (Jan 2022)
        df = web.DataReader(stooq_ticker, 'stooq', start='2022-01-01')

        if not df.empty:
            # IMPORTANT: Stooq returns data REVERSED (Newest first).
            # LSTMs require chronological order (Oldest first).
            df = df.sort_index(ascending=True)

            df.to_csv(file_path)
            print(f"✅ {ticker}: Success")
            success_count += 1
        else:
            print(f"❌ {ticker}: Empty Data")

        # Rate limit safety: Pause briefly between every request
        time.sleep(1.2)

    except Exception as e:
        error_msg = str(e)
        print(f"💥 {ticker} Failed: {error_msg}")

        # If we hit a 429 (Too Many Requests), take a longer break
        if "429" in error_msg:
            print("🛑 Rate limited! Cooling down for 45 seconds...")
            time.sleep(45)

        failed_tickers.append(ticker)

print(f"\n🚀 Finished! Successfully downloaded {success_count} new stocks.")
if failed_tickers:
    print(f"⚠️ Failed tickers: {failed_tickers}")

Starting batch download of 508 stocks via Stooq...
Data will be saved to: C:\Technion\Semester9\Deep Learning Code\Project\Data
✅ MMM: Success
✅ AOS: Success
✅ ABT: Success
✅ ABBV: Success
✅ ACN: Success
✅ ADBE: Success
✅ AMD: Success
✅ AES: Success
✅ AFL: Success
✅ A: Success
✅ APD: Success
✅ ABNB: Success
✅ AKAM: Success
✅ ALB: Success
✅ ARE: Success
✅ ALGN: Success
✅ ALLE: Success
✅ LNT: Success
✅ ALL: Success
✅ GOOGL: Success
✅ GOOG: Success
✅ MO: Success
✅ AMZN: Success
✅ AMCR: Success
✅ AEE: Success
✅ AEP: Success
✅ AXP: Success
✅ AIG: Success
✅ AMT: Success
✅ AWK: Success
✅ AMP: Success
✅ AME: Success
✅ AMGN: Success
✅ APH: Success
✅ ADI: Success
✅ AON: Success
✅ APA: Success
✅ APO: Success
✅ AMAT: Success
✅ APP: Success
✅ APTV: Success
✅ ACGL: Success
✅ ADM: Success
✅ ARES: Success
✅ ANET: Success
✅ AJG: Success
✅ AIZ: Success
✅ T: Success
✅ ATO: Success
✅ ADSK: Success
✅ ADP: Success
✅ AZO: Success
✅ AVB: Success
✅ AVY: Success
✅ AXON: Success
✅ BKR: Success
✅ BALL: Success
✅ 

In [6]:
class StockDataset(Dataset):
    def __init__(self, ticker_symbol, M=5, t_percent=0.05, start="2020-01-01", skip_earnings=True):
        self.M = M  # Modular lookback window (Hyperparameter)

        # 1. Download Price Data
        ticker = yf.Ticker(ticker_symbol)
        df = ticker.history(start=start)
        prices = df['Close'].values
        dates = df.index

        # 2. Handle Earnings Dates (Blacklist)
        blacklisted_dates = set()
        if skip_earnings:
            earnings = ticker.earnings_dates
            if earnings is not None:
                # We skip the entire week (window) surrounding the report
                for report_date in earnings.index:
                    # Convert to date format for comparison
                    d = report_date.date()
                    for offset in range(-2, 3): # Buffer: 2 days before/after
                        blacklisted_dates.add(d + timedelta(days=offset))

        self.X, self.y = [], []

        # 3. Sliding Window Logic (Tutorial 07 - Many-to-One)
        for i in range(len(prices) - M - 1):
            window_dates = dates[i : i + M + 1].date

            # Check if any date in this window is a financial report date
            if skip_earnings and any(wd in blacklisted_dates for wd in window_dates):
                continue # Skip this noisy week

            # Feature: The window of M prices
            self.X.append(prices[i : i + M])

            # Label: 1 if next price > current_price * (1 + t), else 0
            # This is your specific t% jump target logic
            current_price = prices[i + M - 1]
            next_price = prices[i + M]
            label = 1 if (next_price >= current_price * (1 + t_percent)) else 0
            self.y.append(label)

        # Convert to Tensors (Tutorial 05 standards)
        self.X = torch.tensor(np.array(self.X), dtype=torch.float32).unsqueeze(-1)
        self.y = torch.tensor(np.array(self.y), dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# --- Usage in Notebook ---
M = 5 # Start with the previous week
dataset = StockDataset("NVDA", M=M, t_percent=0.02)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"Created {len(dataset)} clean sequences after filtering earnings noise.")

YFRateLimitError: Too Many Requests. Rate limited. Try after a while.

In [3]:
from src.models.stock_lstm import StockLSTM

# 1. Hyperparameters (Tutorial 08 Tuning starts here)
input_dim = 1      # Only 'Close' price for now
hidden_dim = 64    # Number of LSTM units
num_layers = 2     # Stacked LSTM layers
lr = 0.001         # Learning rate

# 2. Setup Device & Model

model = StockLSTM(input_dim, hidden_dim, num_layers).to(device)

# 3. Loss & Optimizer (Tutorial 08 Standard)
# BCELoss is used because we are doing Binary Classification (Jump vs. No Jump)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

print(f"Model initialized on {device}. Ready for training.")

Model initialized on cpu. Ready for training.


In [ ]:
epochs = 50
loss_history = []

model.train()
for epoch in range(epochs):
    epoch_loss = 0
    for batch_X, batch_y in dataloader:
        # Move data to GPU/CPU
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        # 1. Forward pass (Tutorial 07)
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)

        # 2. Backward and optimize (Tutorial 04 Autograd)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(dataloader)
    loss_history.append(avg_loss)

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}')

print("Training Complete.")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(loss_history, label='Training Loss')
plt.title(f'LSTM Training Loss (M={M}, target={dataset.t*100}%)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()